# Youtube Video Summarization

## My First Frontier LLM Project!

Welcome to my first LLM-based project! The goal of this project is to leverage large language models (LLMs) to summarize YouTube videos. Currently, it only supports English transcriptions, so instead of watching the entire video, you can simply read the summary!

## Important Note
Be mindful when testing with longer videos, as they may consume significant resources and could lead to high costs on your ChatGPT bill.
You can switch to Ollama for free usage if you're looking to reduce costs.


In [ ]:
%pip install youtube-transcript-api openai

In [1]:
# imports

import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
import re

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
openai = OpenAI()
youtubeApi = YouTubeTranscriptApi()

In [4]:
class YoutubeVideoID:
    def __init__(self, url):
        self.url = url
        self.video_id = self.extract_video_id(url)

    def extract_video_id(self, url):
        """
        Extracts the YouTube video ID from a given URL.
        Supports both regular and shortened URLs.
        """
        # Regular expression to match YouTube video URL and extract the video ID
        regex = r"(?:https?:\/\/)?(?:www\.)?(?:youtube\.com\/(?:[^\/\n\s]+\/\S+\/|\S*\?v=)|(?:youtu\.be\/))([a-zA-Z0-9_-]{11})"
        match = re.match(regex, url)
        
        if match:
            return match.group(1)
        else:
            raise ValueError("Invalid YouTube URL")

    def __str__(self):
        return f"Video ID: {self.video_id}"

In [33]:
# Example usage
video_url = "https://www.youtube.com/watch?v=OYvlznJ4IZQ"

yt_video = YoutubeVideoID(video_url)
print(yt_video)

Video ID: OYvlznJ4IZQ


In [34]:
def get_transcript(video_id, language='en'):
    try:
        # Try to get the transcript in the desired language
        transcript = youtubeApi.fetch(video_id, languages=[language]).to_raw_data()
        # Join all the 'text' fields into a single string
        return " ".join([item['text'] for item in transcript])
    except Exception as e:
        print(f"Error fetching transcript: {e}")
        return "error"


In [35]:
# Fetch transcript using the video ID
transcript_text = get_transcript(yt_video.video_id)
print(len(transcript_text))
print(transcript_text)

36025
Hi everyone, this is GKCS. In today's video we will see some of
the commonly used terms in the AI space. If you are an engineer
who is building applications, then you will find these terms useful. When communicating with people
within your team or outside. And I think if you know these terms, then it is also easier to learn
the deeper subjects around AI. So by the end of this video,
you'll have a list of terms whose definitions you understand
quite well. And I'll also be linking some references in the description
so that you can dig into them further. Let's start. The first term that you should know about is large language model. Also known as LM. And the definition
of this is a neural network. That is trained to predict the next term. Of an input sequence. For example, if I pass in the query all that glitters to a large language model, then it's going to come up with the response of is not going okay. At which point the complete response
of all that glitters is not gold is retur

In [43]:
# Function to summarize text using ChatGPT
def summarize_text(text):
    try:
        system_prompts = """
        You are a helpful assistant who provides concise and accurate summaries of text. Your task is to:
        
        - Capture the key points of the content.
        - Keep the summary brief and easy to understand.
        - Avoid summarizing overly lengthy texts or breaking them into excessively short summaries.
        - Use bullet points where appropriate to enhance clarity and structure.
        """
        response = openai.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "system", "content": system_prompts},
                {"role": "user", "content": f"Summarize the following text:\n{text}"}
            ],
            max_tokens=30000
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error summarizing text: {e}")
        return None

In [44]:
def split_text(text: str, chunk_size=3000):
    """
    Splits large text into smaller chunks based on the given chunk size.
    Ensures that chunks end with a full stop where possible to maintain sentence integrity.
    
    :param text: str, the text to be split
    :param chunk_size: int, maximum size of each chunk (default 3000 characters)
    :return: list of str, where each str is a chunk of text
    """
    chunks = []
    while len(text) > chunk_size:
        # Find the last full stop within or at the chunk size
        split_point = text.rfind('.', 0, chunk_size + 1)  # +1 to include the period itself if it's at chunk_size
        if split_point == -1:  # No period found within the chunk size
            split_point = chunk_size
        
        # Append the chunk, ensuring we don't strip spaces that might be part of the sentence structure
        chunks.append(text[:split_point + 1] if split_point != chunk_size else text[:chunk_size])
        text = text[split_point + 1:] if split_point != chunk_size else text[chunk_size:]
    
    # Add the remaining text as the final chunk, only strip if there's content
    if text:
        chunks.append(text.strip())
    
    return chunks

transcript_chunks = split_text(transcript_text)

# Now you can summarize each chunk individually
summaries = []
for chunk in transcript_chunks:
    summary = summarize_text(chunk)
    summaries.append(summary)


# Combine the individual summaries into one
full_summary = " ".join(summaries)
display(Markdown(full_summary))


Key Points:

- The video introduces commonly used AI terms useful for engineers developing applications, aiming to improve communication and facilitate deeper learning in AI.
- Large Language Model (LLM): A neural network trained to predict the next word in a sequence (e.g., completing "all that glitters is not gold" from a partial input).
- Tokenization: The process of breaking input text into discrete units (tokens), which may not always correspond to words split by spaces. Tokenization helps models process and understand language structure efficiently.
- Vectors: Represent tokens (words or phrases) as points in an n-dimensional space, where similar meanings are placed closer together. This allows the model to capture and manipulate word meanings as coordinates.
- More terms and detailed explanations are covered in the video, with reference links provided for further exploration. - Words are represented as vectors in a multi-dimensional space through vectorization, so similar words are close together and opposites are far apart.
- Large Language Models (LLMs) can tokenize input text and understand the inherent meaning of words based on these vectors.
- A key challenge is understanding ambiguous words (e.g., "apple") whose meaning depends on context.
- LLMs use the "attention" mechanism to derive word meaning from surrounding words, helping resolve ambiguity by adjusting vectors according to context.
- This mechanism allows models to comprehend nuanced meanings effectively.
- The breakthrough attention mechanism was introduced in 2017, but became widely known after GPT-2's release in 2022. - Large language models (LLMs) excel at generating high-quality, context-aware, human-like responses.
- Major progress came in 2017 with the rise of self-supervised learning.
- Self-supervised learning allows models to learn from structure within the data itself, without explicit human labeling.
- Example: By masking parts of a sequence, the model learns to predict missing information based on inherent patterns.
- Standard supervised learning relies on human-annotated input/output pairs, but self-supervised methods make training data generation much cheaper by utilizing raw text.
- During training, models are penalized for incorrect predictions (increasing loss), and rewarded (stable weights) for correct ones, refining their internal representations over time. - Self-supervised learning involves training AI models using existing data without human intervention, making them highly scalable.
- In self-supervised text models, the system creates its own challenges, such as predicting missing words; for images, it predicts missing patches.
- The transformer is a specific algorithm used in large language models (LLMs) to predict the next token in a sequence, but it is not synonymous with LLMs.
- Transformers work by passing input tokens through multiple attention layers and feedforward neural networks, progressively understanding deeper relationships and meanings in the data.
- Attention layers help disambiguate meanings (e.g., distinguishing homonyms, detecting implications or emotions) at different levels.
- Transformers are often stacked in multiple layers (dozens to hundreds) to improve model performance.
- The transformer is like the engine of a car (the LLM), and in theory, other architectures (like diffusion models) could replace it in the future. - Fine-tuning is the process of adapting a trained base language model to give domain-specific answers (e.g., medical or financial terms).
- The base model is first trained generally, then further trained (fine-tuned) with specific questions and answers to tailor its responses and behavior.
- Fine-tuning penalizes undesirable answers and encourages direct, context-appropriate responses.
- A single base model can be fine-tuned multiple ways, producing specialized models for different companies or tasks.
- Few-shot prompting is another technique where, during inference, example queries and answers are given to the model along with a new query; this helps the model respond more accurately by putting examples in context, improving response quality.
- The next topic mentioned is retrieval-augmented generation (RAG). - The AI field is evolving rapidly, with some claiming retrieval augmented generation (RAG) is becoming outdated.
- RAG works by sending both user queries and relevant company documents (like policies or examples) to a large language model (LLM) to improve response quality.
- Relevant documents are fetched in real-time and combined with user queries and example prompts to provide context to the LLM.
- The choice of how to store and retrieve documents (graph databases, vector databases, or in-memory cache) is flexible, though vector databases are common as they enable similarity searches.
- The main goal is to enrich the LLM’s input with as much relevant context as possible to generate high-quality responses.
- Vector databases help match user queries with the most relevant documents, even if exact keywords are not present, by using similarity search. - Semantic similarity can be measured using vectors: similar words (like "upset" and "low rating") are close in vector space.
- Documents are stored in a vector database, which allows efficient similarity searches to find relevant context for queries.
- When a user submits a query, the system finds the closest matching documents (based on vectors) to provide context for a large language model (LLM).
- Vector databases function as "black boxes" where documents are stored and quickly retrieved to support LLM tasks.
- When relevant context exists outside the internal database, the Model Context Protocol (MCP) enables the LLM to fetch external information (e.g., flight details from airlines).
- The LLM, via an MCP client, identifies when external data is needed, requests it from external servers (MCP servers), and incorporates the real-time data into its decision process.
- This approach allows the LLM to provide richer, real-time, and context-aware responses, ultimately improving user satisfaction. - MCP clients can now execute tasks automatically, making LMS (Language Model Systems) more powerful.
- These advancements are part of "context engineering," which includes:
  - Few-shot prompting: Providing examples to LMs.
  - Retrieval-augmented generation: Adding relevant documents from databases to user queries.
  - Model context protocol (MCP): Allowing external server actions.
- New challenges in context engineering:
  - Handling user preferences.
  - Summarizing prompts (context summarization) to stay within LLM input limits, using methods like:
    - Sliding windows for recent chats.
    - Summarizing older conversations.
    - Keyword extraction.
- Context summarization often uses smaller models before sending information to larger, more expensive LLMs.
- Difference from prompt engineering:
  - Prompt engineering handles single, stateless prompts.
  - Context engineering tracks evolving user preferences and chat history (stateful and long-term).
- Introduction of agents:
  - Agents are long-running processes capable of managing various tasks (e.g., booking travel, handling email) based on user preferences.
  - Agents can interact with LLMs, external systems, and other agents. - Reinforcement learning is a method for training models to behave in desired ways by assigning rewards or penalties (e.g., +1 for good responses, -1 for bad ones).
- In practice, given a user query, the model generates multiple responses; humans select the better one, reinforcing that path with positive feedback.
- Each model output can be mapped as a path in an n-dimensional vector space, with positive or negative scores assigned along the way.
- Over time, this creates a landscape of preferred (positive) and avoided (negative) paths, guiding the model to generate better outputs.
- The process is akin to hill climbing—optimizing for user satisfaction using reinforcement signals (human feedback).
- This method, known as Reinforcement Learning with Human Feedback (RLHF), teaches the model to produce responses that make users happy.
- The process parallels natural conditioning, such as Pavlov’s dog experiment, reinforcing desirable behavior through reward. - Human intelligence cannot be fully modeled by reinforcement learning alone.
- Example: Given a fair coin, humans use their understanding of probability and physics to estimate a 50/50 chance, regardless of observed streaks, whereas reinforcement learning relies only on observed outcomes to make predictions.
- Humans build mental models, allowing deeper reasoning, while reinforcement learning evaluates only based on past outcomes.
- Reinforcement learning is still a powerful tool for making models smarter.
- "Chain of thought" is a training technique where a model is taught to solve problems step-by-step, improving reasoning and response quality.
- This approach allows models to handle harder problems by taking more steps and fewer steps for easier problems, as observed in research.
- Reasoning models can use chain of thought or other algorithms for problem-solving. - Reasoning models can solve problems step-by-step, using structures like “tree of thought” or “graph of thought.”
- Modern models include deepseek, OpenAI’s latest (O1, O3), and multimodal models.
- Multimodal models process multiple types of input (text, images, video), enabling them to generate and analyze content beyond text, such as counting objects in images or editing videos.
- These models have applications in marketing, advertising, and content creation, potentially reducing the cost of high-quality media production.
- Multimodal models outperform text-only models by having a deeper understanding when trained on both images and textual data.
- There is a shift towards smaller, company-specific models (Small Language Models or SLMs) for more control and privacy; these have fewer parameters (3M–300M) compared to large language models (3B–300B).
- SLMs are often used for specific business tasks, like managing customer queries or sales, and can perform well with focused training. Summary:

- Companies train smaller language models (LLMs) on their own proprietary data for specific use cases (e.g., sales, weather prediction).
- NASA would prioritize weather analysis over sales capabilities in its models.
- Model distillation is used to train smaller language models: a large model (teacher) produces outputs, and a small model (student) learns to mimic these outputs by adjusting its internal weights.
- Small models are faster, cheaper, and easier to deploy than large ones, though they may be less accurate.
- Quantization further reduces a model's memory and inference costs by compressing neural network weights (e.g., from 32-bit to 8-bit), saving resources mainly during inference.
- Training costs remain unchanged by quantization; the reduction benefits are realized in production use.
- Understanding these concepts (distillation, quantization, etc.) is vital for effective communication in the engineering field. - Mastering the engineering course content gives you a true understanding of how models work.
- This knowledge helps you distinguish between genuine information and hype or misinformation in the field.
- The speaker thanks viewers and concludes the video.